In [ ]:
# Install required packages
!pip install kaggle ultralytics opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 68.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstal

In [ ]:
import os
import zipfile
import shutil
from glob import glob
from sklearn.model_selection import train_test_split
from ultralytics import YOLO


In [ ]:
os.environ['KAGGLEHUB_CACHE'] = '/content/'

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mikoajkoek/traffic-road-object-detection-polish-12k")

print("Path to dataset files:", path)

Path to dataset files: /content/datasets/mikoajkoek/traffic-road-object-detection-polish-12k/versions/4


In [ ]:
# Base dataset path
BASE = "/content/datasets/mikoajkoek/traffic-road-object-detection-polish-12k/versions/4/road_detection/road_detection"
SPLITS = ["train", "valid", "test"]
MAX_CLASS = 10  # valid IDs: 0–10 (11 classes)

for split in SPLITS:
    lbl_dir = os.path.join(BASE, split, "labels")
    img_dir = os.path.join(BASE, split, "images")
    for lbl in glob(f"{lbl_dir}/*.txt"):
        lines = open(lbl).read().splitlines()
        # if *any* label line has class > MAX_CLASS, remove label & image
        if any(int(l.split()[0]) > MAX_CLASS for l in lines):
            print("Removing invalid:", lbl)
            os.remove(lbl)
            stem = os.path.splitext(os.path.basename(lbl))[0]
            for ext in (".jpg", ".png"):
                fp = os.path.join(img_dir, stem + ext)
                if os.path.exists(fp):
                    os.remove(fp)

In [ ]:
dataset_yaml = """
path: /content/datasets/mikoajkoek/traffic-road-object-detection-polish-12k/versions/4/road_detection/road_detection
train: train/images
val: valid/images
test:  test/images
names:
  0: Car
  1: Different-Traffic-Sign
  2: Green-Traffic-Light
  3: Motorcycle
  4: Pedestrian
  5: Pedestrian-Crossing
  6: Prohibition-Sign
  7: Red-Traffic-Light
  8: Speed-Limit-Sign
  9: Truck
  10: Warning-Sign
"""

with open("traffic_polish.yaml", "w") as f:
    f.write(dataset_yaml)


In [ ]:
# Load a pre-trained YOLOv8 model
model = YOLO("yolov8s.pt")


In [ ]:
# Train with checkpointing and evaluation enabled
model.train(
    data="/content/traffic_polish.yaml",
    epochs=10,
    patience=10,
    imgsz=640,
    batch=16,
    name="yolov8s_traffic_polish",
    save=True,
    save_period=10,  # Save every 10 epochs
    val=True,
    workers=4,
    device=0  # use "cpu" if no GPU is available
)


Ultralytics 8.3.112 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
engine/trainer: task=detect, mode=train, model=yolov8s.pt, data=/content/traffic_polish.yaml, epochs=10, time=None, patience=10, batch=16, imgsz=640, save=True, save_period=10, cache=False, device=0, workers=4, project=None, name=yolov8s_traffic_polish6, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True,

train: Scanning /content/datasets/mikoajkoek/traffic-road-object-detection-polish-12k/versions/4/road_detection/road_detection/train/labels.cache... 10474 images, 0 backgrounds, 0 corrupt: 100%|██████████| 10474/10474 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 808.5±133.1 MB/s, size: 879.3 KB)


val: Scanning /content/datasets/mikoajkoek/traffic-road-object-detection-polish-12k/versions/4/road_detection/road_detection/valid/labels.cache... 935 images, 0 backgrounds, 0 corrupt: 100%|██████████| 935/935 [00:00<?, ?it/s]


...

                   all        935       8981      0.821      0.673       0.73      0.432



10 epochs completed in 0.353 hours.
Optimizer stripped from runs/detect/yolov8s_traffic_polish6/weights/last.pt, 22.5MB
Optimizer stripped from runs/detect/yolov8s_traffic_polish6/weights/best.pt, 22.5MB

Validating runs/detect/yolov8s_traffic_polish6/weights/best.pt...
Ultralytics 8.3.112 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
Model summary (fused): 72 layers, 11,129,841 parameters, 0 gradients, 28.5 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 30/30 [00:06<00:00,  4.35it/s]


                   all        935       8981      0.822      0.673       0.73      0.432
                   Car        906       3593      0.887      0.822      0.893      0.628
Different-Traffic-Sign        671       2479      0.859      0.709      0.793      0.455
   Green-Traffic-Light        105        219      0.723      0.703      0.697      0.401
            Motorcycle         28         28      0.887      0.838      0.852      0.501
            Pedestrian        202        578      0.769      0.604       0.68      0.311
   Pedestrian-Crossing        178        271      0.717       0.48      0.547      0.303
      Prohibition-Sign        209        280      0.891      0.718      0.793      0.456
     Red-Traffic-Light        238        606      0.935      0.909      0.906      0.587
      Speed-Limit-Sign        115        149      0.736      0.443      0.501      0.277
                 Truck        221        359      0.788      0.588       0.68       0.44
          Warning-Sig

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7da7d04fe050>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.

In [ ]:
# Load best model from training
best_model_path = "runs/detect/yolov8s_traffic_polish6/weights/best.pt"
model = YOLO(best_model_path)


In [ ]:
# Evaluate the model on the test set
metrics = model.val(data="/content/traffic_polish.yaml")

# Access accuracy from the metrics dictionary
accuracy = metrics.box.map

# Print the accuracy
print(f"Accuracy: {accuracy}")


Ultralytics 8.3.112 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
Model summary (fused): 72 layers, 11,129,841 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3182.5±753.5 MB/s, size: 824.2 KB)


val: Scanning /content/datasets/mikoajkoek/traffic-road-object-detection-polish-12k/versions/4/road_detection/road_detection/valid/labels.cache... 935 images, 0 backgrounds, 0 corrupt: 100%|██████████| 935/935 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 59/59 [00:07<00:00,  7.42it/s]


                   all        935       8981      0.824      0.673       0.73      0.435
                   Car        906       3593      0.888      0.823      0.894      0.629
Different-Traffic-Sign        671       2479      0.862      0.708      0.794      0.457
   Green-Traffic-Light        105        219      0.724      0.703      0.699      0.405
            Motorcycle         28         28      0.886      0.836      0.852      0.518
            Pedestrian        202        578       0.77      0.607      0.682      0.314
   Pedestrian-Crossing        178        271      0.718       0.48      0.543      0.304
      Prohibition-Sign        209        280      0.889      0.714      0.792      0.457
     Red-Traffic-Light        238        606      0.933      0.909      0.909      0.587
      Speed-Limit-Sign        115        149      0.736      0.443      0.502      0.281
                 Truck        221        359      0.794      0.588      0.681      0.437
          Warning-Sig

In [ ]:
# Inference on a new image
results = model("data/traffic_polish_12k/images/val/example.jpg")  # Change image path accordingly
results.show()  # Displays with bounding boxes
results.save(filename="prediction.jpg")  # Saves annotated image


In [ ]:
# prompt: zip the folder

import zipfile
import os

def zip_folder(folder_path, zip_path):
  """Zips a folder and its contents.

  Args:
    folder_path: The path to the folder to zip.
    zip_path: The path to the output zip file.
  """
  with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(folder_path):
      for file in files:
        zipf.write(os.path.join(root, file),
                   os.path.relpath(os.path.join(root, file),
                                   os.path.join(folder_path, '..')))

# Example usage:
zip_folder("/content/runs/detect/yolov8s_traffic_polish6", "yolov8s_traffic_polish6.zip")


In [ ]:
!gdown "https://drive.google.com/uc?id=1rjBn8Fl1E_9d0EMVtL24S9aNQOJAveR5&confirm=t"


Downloading...
From: https://drive.google.com/uc?id=1rjBn8Fl1E_9d0EMVtL24S9aNQOJAveR5&confirm=t
To: /content/test3.mp4
100% 4.98M/4.98M [00:00<00:00, 42.4MB/s]


In [ ]:
from google.colab import files

# Upload a video file
uploaded = files.upload()       # select .mp4 or .avi
video = next(iter(uploaded))    # get the filename



In [ ]:
# Run inference & save annotated video
results = model.predict(
    source="/content/test3.mp4",
    save=True,
    imgsz=640,
    conf=0.5,
    iou=0.5
)


WARNING ⚠️ 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/508) /content/test3.mp4: 384x640 1 Car, 55.0ms
video 1/1 (frame 2/508) /content/test3.mp4: 384x640 (no detections), 8.3ms
video 1/1 (frame 3/508) /content/test3.mp4: 384x640 (no detections), 7.6ms
video 1/1 (frame 4/508) /content/test3.mp4: 384x640 (no detections), 7.5ms
video 1/1 (frame 5/508) /content/test3.mp4: 384x640 (no detections), 7.6ms
video 1/1 (frame 6/508) /content/test3.mp4: 384x640 (no detections), 7.9ms
video 1/1 (fram